# Quickstart

A step is a function. Its arguments are the columns it reads and its name is the column it writes. `param()` marks a policy value you can retune without changing code.

In [1]:
import polars as pl
from decider import flow, param

def ratio(income: float, debt: float) -> float:
    return debt / income

def affordable(ratio: float, limit: float = param(0.4)) -> bool:
    return ratio <= limit

pipeline = flow(ratio, affordable, name="afford")

## Run a frame

In [2]:
df = pl.DataFrame({"income": [1000.0, 500.0], "debt": [200.0, 400.0]})
print(pipeline.run(df))

shape: (2, 3)
┌────────┬───────┬────────────┐
│ income ┆ debt  ┆ affordable │
│ ---    ┆ ---   ┆ ---        │
│ f64    ┆ f64   ┆ bool       │
╞════════╪═══════╪════════════╡
│ 1000.0 ┆ 200.0 ┆ true       │
│ 500.0  ┆ 400.0 ┆ false      │
└────────┴───────┴────────────┘


## Score one record

Compile once, then score a dict.

In [3]:
fused = pipeline.compile("fused")
fused.score({"income": 1000.0, "debt": 200.0})

{'income': 1000.0, 'debt': 200.0, 'affordable': True}

## Retune a param

The params document is keyed by step path. Nothing is rebuilt.

In [4]:
print(pipeline.run(df, params={"afford": {"affordable": {"limit": 0.9}}}))

shape: (2, 3)
┌────────┬───────┬────────────┐
│ income ┆ debt  ┆ affordable │
│ ---    ┆ ---   ┆ ---        │
│ f64    ┆ f64   ┆ bool       │
╞════════╪═══════╪════════════╡
│ 1000.0 ┆ 200.0 ┆ true       │
│ 500.0  ┆ 400.0 ┆ true       │
└────────┴───────┴────────────┘
